# Additional Regressions

Labor+railroad cross-sectional, biographical controls, panel regressions, editor/publisher split, and WLS robustness checks.

In [15]:
import pandas as pd
import numpy as np
import sqlite3
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col
import warnings
warnings.filterwarnings('ignore')

OE_PATH = '../data/intermediate/personnel_coding/owners_and_editors.csv'

df = pd.read_csv('intermediate/analysis_sample.csv')
counts = pd.read_csv('intermediate/sentiment_counts.csv')
paper_year_rr = pd.read_csv('intermediate/paper_year_rr.csv')
person_rr = pd.read_csv('intermediate/person_rr.csv')
oe = pd.read_csv(OE_PATH)

# Build cross-sectional df (newspaper-level, all labor articles)
df_xs = (
    df.groupby('issn')
    .agg(anti_labor=('anti_labor', 'sum'),
         total_labor=('total_labor', 'sum'),
         railroad_interest=('railroad_interest', 'max'),
         railroad_interest_level=('railroad_interest_level', 'max'),
         person_republican=('person_republican', 'max'),
         log_county_pop=('log_county_pop', 'mean'))
    .reset_index()
)
df_xs['anti_labor_intensity'] = df_xs['anti_labor'] / df_xs['total_labor']

print(f'Loaded analysis sample: {len(df)} newspaper-years, {len(df_xs)} newspapers')

Loaded analysis sample: 319 newspaper-years, 57 newspapers


### Cross-Sectional: Labor+Railroad Articles Only

Restricted to articles classified as both labor- and railroad-related. Newspapers must have ≥20 labor+railroad articles across the owner's tenure.

In [16]:
# Collapse to newspaper level using labor+railroad articles only (min 20 per tenure)
agg_cols = dict(anti_labor_rr=('anti_labor_rr', 'sum'), total_labor_rr=('total_labor_rr', 'sum'),
                railroad_interest=('railroad_interest', 'max'), person_republican=('person_republican', 'max'),
                log_county_pop=('log_county_pop', 'mean'), military_service=('military_service', 'max'),
                education_level=('education_level', 'max'), age=('age', 'mean'),
                public_office_sought=('public_office_sought', 'max'))
df_xs_rr = (df.groupby('issn').agg(**agg_cols).reset_index()
            .query('total_labor_rr >= 25'))
df_xs_rr['anti_labor_intensity_rr'] = df_xs_rr['anti_labor_rr'] / df_xs_rr['total_labor_rr']

rr_a = smf.ols('anti_labor_intensity_rr ~ railroad_interest', data=df_xs_rr).fit(cov_type='HC3')
rr_b = smf.ols('anti_labor_intensity_rr ~ railroad_interest + person_republican',
               data=df_xs_rr.dropna(subset=['person_republican'])).fit(cov_type='HC3')
rr_c = smf.ols('anti_labor_intensity_rr ~ railroad_interest + person_republican + log_county_pop',
               data=df_xs_rr.dropna(subset=['person_republican', 'log_county_pop'])).fit(cov_type='HC3')

print(f"Cross-sectional \u2014 labor+railroad articles only (\u226520 per tenure), OLS HC3")
print(summary_col([rr_a, rr_b, rr_c],
    model_names=['(0a) Bivariate', '(0b) + Party', '(0c) + Party + Pop'], stars=True,
    info_dict={'N': lambda m: m.nobs, 'R\u00b2': lambda m: round(m.rsquared, 3)},
    regressor_order=['railroad_interest', 'person_republican', 'log_county_pop', 'Intercept']))

Cross-sectional — labor+railroad articles only (≥20 per tenure), OLS HC3

                  (0a) Bivariate (0b) + Party (0c) + Party + Pop
----------------------------------------------------------------
railroad_interest 0.0823*        0.1083**     0.1011            
                  (0.0492)       (0.0536)     (0.0710)          
person_republican                -0.0458      -0.0272           
                                 (0.0516)     (0.0745)          
log_county_pop                                -0.0080           
                                              (0.0426)          
Intercept         0.4150***      0.4209***    0.4980            
                  (0.0380)       (0.0415)     (0.4307)          
R-squared         0.1143         0.2251       0.1796            
R-squared Adj.    0.0774         0.1436       0.0349            
N                 26.0000        22.0000      21.0000           
R²                0.1140         0.2250       0.1800            
Standard errors 

### Cross-Sectional: Ordinal Railroad Interest Level

Uses an ordinal ranking of railroad connection strength instead of a binary indicator:
0 = no interest, 1 = family connection, 2 = donation, 3 = professional services, 4 = stockholder, 5 = board member, 6 = company owner.

In [17]:
# Cross-sectional with ordinal railroad interest level
print(f"Railroad interest level distribution (newspaper-level):")
print(df_xs['railroad_interest_level'].value_counts().sort_index())
print()

ol_a = smf.ols('anti_labor_intensity ~ railroad_interest_level', data=df_xs).fit(cov_type='HC3')
ol_b = smf.ols('anti_labor_intensity ~ railroad_interest_level + person_republican',
               data=df_xs.dropna(subset=['person_republican'])).fit(cov_type='HC3')
ol_c = smf.ols('anti_labor_intensity ~ railroad_interest_level + person_republican + log_county_pop',
               data=df_xs.dropna(subset=['person_republican', 'log_county_pop'])).fit(cov_type='HC3')

print("Cross-sectional (ordinal railroad interest), OLS HC3")
print(summary_col([ol_a, ol_b, ol_c],
    model_names=['(0a) Bivariate', '(0b) + Party', '(0c) + Party + Pop'], stars=True,
    info_dict={'N': lambda m: m.nobs, 'R\u00b2': lambda m: round(m.rsquared, 3)},
    regressor_order=['railroad_interest_level', 'person_republican', 'log_county_pop', 'Intercept']))

Railroad interest level distribution (newspaper-level):
railroad_interest_level
0    30
1     8
3     3
4     6
5     8
6     2
Name: count, dtype: int64

Cross-sectional (ordinal railroad interest), OLS HC3

                        (0a) Bivariate (0b) + Party (0c) + Party + Pop
----------------------------------------------------------------------
railroad_interest_level 0.0239***      0.0234**     0.0257**          
                        (0.0084)       (0.0097)     (0.0110)          
person_republican                      0.0624*      0.0658*           
                                       (0.0368)     (0.0393)          
log_county_pop                                      -0.0440           
                                                    (0.0274)          
Intercept               0.3230***      0.2885***    0.7488***         
                        (0.0188)       (0.0273)     (0.2857)          
R-squared               0.1288         0.1662       0.2556            
R-squared 

### Cross-Sectional with Biographical Controls (all labor articles)

Starts from railroad interest + party, then adds one biographical control at a time.

In [18]:
# Cross-sectional with biographical controls (all labor articles, one control at a time)
bio_agg = df.groupby('issn').agg(
    military_service=('military_service', 'max'), education_level=('education_level', 'max'),
    age=('age', 'mean'), public_office_sought=('public_office_sought', 'max')).reset_index()
df_xs_bio = df_xs.merge(bio_agg, on='issn')

base = 'anti_labor_intensity ~ railroad_interest + person_republican'
controls = ['military_service', 'education_level', 'age', 'public_office_sought']
models, col_names = [], []
for ctrl in controls:
    models.append(smf.ols(f'{base} + {ctrl}', data=df_xs_bio.dropna(subset=['person_republican', ctrl])).fit(cov_type='HC3'))
    col_names.append(f'+ {ctrl}')

print("Cross-sectional with biographical controls (all labor articles), OLS HC3")
print(summary_col(models, model_names=col_names, stars=True,
    info_dict={'N': lambda m: m.nobs, 'R\u00b2': lambda m: round(m.rsquared, 3)},
    regressor_order=['railroad_interest', 'person_republican'] + controls + ['Intercept']))

Cross-sectional with biographical controls (all labor articles), OLS HC3

                     + military_service + education_level   + age   + public_office_sought
------------------------------------------------------------------------------------------
railroad_interest    0.0963**           0.1152*           0.0879**  0.0939**              
                     (0.0448)           (0.0601)          (0.0443)  (0.0434)              
person_republican    0.0577             0.0470            0.0639    0.0583                
                     (0.0387)           (0.0549)          (0.0409)  (0.0403)              
military_service     0.0511                                                               
                     (0.0421)                                                             
education_level                         -0.0295                                           
                                        (0.0250)                                          
age             

### Panel Regressions (newspaper-year level, OLS, HC3 SEs)

One observation per newspaper-year. Progressively adds year fixed effects, person-level political affiliation, and county population controls.

In [19]:
# Panel OLS regressions (HC3 SEs)
m1 = smf.ols('anti_labor_intensity ~ railroad_interest', data=df).fit(cov_type='HC3')
m2 = smf.ols('anti_labor_intensity ~ railroad_interest + C(year)', data=df).fit(cov_type='HC3')

df_pol = df.dropna(subset=['person_republican']).copy()
m3 = smf.ols('anti_labor_intensity ~ railroad_interest + C(year) + person_republican',
             data=df_pol).fit(cov_type='HC3')

df_pol_pop = df.dropna(subset=['person_republican', 'log_county_pop']).copy()
m4 = smf.ols('anti_labor_intensity ~ railroad_interest + C(year) + person_republican + log_county_pop',
             data=df_pol_pop).fit(cov_type='HC3')

df_pol_pop_np = df_pol_pop.dropna(subset=['n_other_papers']).copy()
m5 = smf.ols('anti_labor_intensity ~ railroad_interest + C(year) + person_republican + log_county_pop + n_other_papers',
             data=df_pol_pop_np).fit(cov_type='HC3')

print(f"N: full={len(df)} | +party={len(df_pol)} | +party+pop={len(df_pol_pop)} | +n_papers={len(df_pol_pop_np)}")
print()
print(summary_col(
    [m1, m2, m3, m4, m5],
    model_names=['(1) Bivariate', '(2) Year FE', '(3) + Party', '(4) + Pop', '(5) + N papers'],
    stars=True,
    info_dict={'N': lambda m: m.nobs, 'R\u00b2': lambda m: round(m.rsquared, 3)},
    regressor_order=['railroad_interest', 'person_republican', 'log_county_pop', 'n_other_papers', 'Intercept']))

N: full=319 | +party=282 | +party+pop=276 | +n_papers=276


                  (1) Bivariate (2) Year FE (3) + Party (4) + Pop (5) + N papers
--------------------------------------------------------------------------------
railroad_interest 0.0624**      0.0595**    0.0516*     0.0603**  0.0572**      
                  (0.0253)      (0.0245)    (0.0264)    (0.0281)  (0.0285)      
person_republican                           0.0923***   0.1007*** 0.0977***     
                                            (0.0253)    (0.0273)  (0.0274)      
log_county_pop                                          -0.0211*  -0.0290**     
                                                        (0.0113)  (0.0143)      
n_other_papers                                                    0.0002        
                                                                  (0.0001)      
Intercept         0.2940***     0.1965***   0.1769***   0.3985*** 0.4845***     
                  (0.0162)      (0.0570)    (0.05

### Editor vs Publisher Split

Runs the main specification (railroad_interest + year FE) separately for newspaper-years where the coded person is an editor vs a publisher.

In [13]:
# Split by role: editors only vs publishers only
df_editors = df[df['has_editor'] == 1].copy()
df_publishers = df[df['has_publisher'] == 1].copy()

print(f"Newspaper-years with editor:    {len(df_editors)}")
print(f"Newspaper-years with publisher: {len(df_publishers)}")

m4_ed = smf.ols('anti_labor_intensity ~ railroad_interest + C(year)', data=df_editors).fit(cov_type='HC3')
m4_pub = smf.ols('anti_labor_intensity ~ railroad_interest + C(year)', data=df_publishers).fit(cov_type='HC3')

print()
print(summary_col(
    [m4_ed, m4_pub],
    model_names=['Editors only', 'Publishers only'],
    stars=True,
    info_dict={'N': lambda m: m.nobs, 'R\u00b2': lambda m: round(m.rsquared, 3)},
    regressor_order=['railroad_interest', 'Intercept']))

Newspaper-years with editor:    280
Newspaper-years with publisher: 251


                  Editors only Publishers only
----------------------------------------------
railroad_interest 0.0469*      0.0644**       
                  (0.0270)     (0.0309)       
Intercept         0.1968***    0.1966***      
                  (0.0612)     (0.0665)       
C(year)[T.1871]   -0.0176      -0.0418        
                  (0.0703)     (0.0774)       
C(year)[T.1872]   0.1823**     0.1498         
                  (0.0912)     (0.0989)       
C(year)[T.1873]   0.0575       0.0411         
                  (0.0753)     (0.0788)       
C(year)[T.1876]   0.0574       0.0733         
                  (0.1032)     (0.1154)       
C(year)[T.1877]   0.3043***    0.2850***      
                  (0.0738)     (0.0780)       
C(year)[T.1878]   0.0165       0.0187         
                  (0.0675)     (0.0726)       
C(year)[T.1879]   0.0932       0.1065         
                  (0.0836)     (0

### Robustness: WLS + Clustered Standard Errors

Re-runs the main specifications using WLS (weighted by total_labor) and clustering standard errors at the newspaper level.

In [14]:
# WLS weighted by total_labor, SEs clustered by newspaper
cluster_kw = {'groups': df['issn']}
cluster_kw_pol = {'groups': df_pol['issn']}
cluster_kw_pop = {'groups': df_pol_pop['issn']}

w1 = smf.wls('anti_labor_intensity ~ railroad_interest',
             weights=df['total_labor'], data=df).fit(cov_type='cluster', cov_kwds=cluster_kw)
w2 = smf.wls('anti_labor_intensity ~ railroad_interest + C(year)',
             weights=df['total_labor'], data=df).fit(cov_type='cluster', cov_kwds=cluster_kw)
w3 = smf.wls('anti_labor_intensity ~ railroad_interest + C(year) + person_republican',
             weights=df_pol['total_labor'], data=df_pol).fit(cov_type='cluster', cov_kwds=cluster_kw_pol)
w4 = smf.wls('anti_labor_intensity ~ railroad_interest + C(year) + person_republican + log_county_pop',
             weights=df_pol_pop['total_labor'], data=df_pol_pop).fit(cov_type='cluster', cov_kwds=cluster_kw_pop)

print("WLS (weighted by total_labor) + newspaper-clustered SEs")
print()
print(summary_col(
    [w1, w2, w3, w4],
    model_names=['(1) Bivariate', '(2) Year FE', '(3) Year+Party', '(4) +County Pop'],
    stars=True,
    info_dict={'N': lambda m: m.nobs, 'Clusters': lambda m: len(m.cov_kwds["groups"].unique()),
               'R\u00b2': lambda m: round(m.rsquared, 3)},
    regressor_order=['railroad_interest', 'person_republican', 'log_county_pop', 'Intercept']))

WLS (weighted by total_labor) + newspaper-clustered SEs


                  (1) Bivariate (2) Year FE (3) Year+Party (4) +County Pop
--------------------------------------------------------------------------
railroad_interest 0.0592***     0.0465**    0.0523***      0.0470**       
                  (0.0202)      (0.0182)    (0.0188)       (0.0235)       
person_republican                           0.0421**       0.0465*        
                                            (0.0181)       (0.0246)       
log_county_pop                                             0.0033         
                                                           (0.0094)       
Intercept         0.3040***     0.2019***   0.1732***      0.1283         
                  (0.0125)      (0.0328)    (0.0305)       (0.0975)       
C(year)[T.1871]                 0.0486      0.0592         0.0624         
                                (0.0333)    (0.0391)       (0.0389)       
C(year)[T.1872]                 0.0648    